# 1. Khởi tạo môi trường và thư viện


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


# 5. Hàm đánh giá (Metrics Evaluator)


In [ ]:
def compute_metrics(y_true_mod, y_pred_mod, y_true_head, y_pred_head, compounds):
    # 1. Main metric: Overall Spearman Correlation
    spearman_mod, _ = spearmanr(y_true_mod, y_pred_mod)
    spearman_head, _ = spearmanr(y_true_head, y_pred_head)
    main_spearman = (spearman_mod + spearman_head) / 2.0
    
    # 2. Sub metric: Context Variance Correlation
    df_eval = pd.DataFrame({
        'comp': compounds,
        'true_mod': y_true_mod,
        'pred_mod': y_pred_mod,
        'true_head': y_true_head,
        'pred_head': y_pred_head
    })
    
    comp_var = df_eval.groupby('comp').agg({
        'true_mod': 'std', 'pred_mod': 'std',
        'true_head': 'std', 'pred_head': 'std'
    }).fillna(0)
    
    var_spearman_mod, _ = spearmanr(comp_var['true_mod'], comp_var['pred_mod'])
    var_spearman_head, _ = spearmanr(comp_var['true_head'], comp_var['pred_head'])
    sub_var_spearman = (np.nan_to_num(var_spearman_mod) + np.nan_to_num(var_spearman_head)) / 2.0
    
    # 3. Auxiliary metric: RMSE
    rmse_mod = np.sqrt(mean_squared_error(y_true_mod, y_pred_mod))
    rmse_head = np.sqrt(mean_squared_error(y_true_head, y_pred_head))
    avg_rmse = (rmse_mod + rmse_head) / 2.0
    
    return {
        'Spearman_Mod': spearman_mod,
        'Spearman_Head': spearman_head,
        'Main_Spearman': main_spearman,
        'Var_Spearman_Mod': var_spearman_mod,
        'Var_Spearman_Head': var_spearman_head,
        'Sub_Var_Spearman': sub_var_spearman,
        'RMSE_Mod': rmse_mod,
        'RMSE_Head': rmse_head,
        'Avg_RMSE': avg_rmse
    }


# 8. Phương pháp 3: BERT / DeBERTa có giám sát (Fine-tuning)


In [ ]:
class CompoundDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ctx = str(row['Context'])
        comp = str(row['Compound']).strip()
        parts = comp.split()
        m_word, h_word = parts[0], parts[1] if len(parts) >= 2 else ""
        
        marked_comp = f"<m>{m_word}</m> <h>{h_word}</h>"
        pattern = re.compile(re.escape(comp), re.IGNORECASE)
        marked_ctx = pattern.sub(marked_comp, ctx, count=1)
        
        inputs = self.tokenizer(
            marked_ctx,
            comp,
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )
        
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        if not self.is_test:
            item['targets'] = torch.tensor([row['Mod'], row['Head']], dtype=torch.float)
        return item

class DebertaRegressor(nn.Module):
    def __init__(self, model_name='microsoft/deberta-v3-base', num_added_tokens=4):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.head = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Linear(128, 2)
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        if token_type_ids is not None:
            outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        else:
            outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_rep = outputs.last_hidden_state[:, 0, :]
        logits = self.head(cls_rep)
        return torch.sigmoid(logits) * 5.0


# 9. Huấn luyện và đánh giá


In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0.0
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['targets'].to(device)
        
        token_type_ids = batch.get('token_type_ids', None)
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)
            
        preds = model(input_ids, attention_mask, token_type_ids)
        loss = criterion(preds, targets)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    all_preds = []
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch.get('token_type_ids', None)
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)
            
        preds = model(input_ids, attention_mask, token_type_ids)
        all_preds.append(preds.cpu().numpy())
    return np.vstack(all_preds)

def run_fine_tuning(train_df, val_df, model_name='microsoft/deberta-v3-base', epochs=3, batch_size=16, lr=2e-5):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    special_tokens = {'additional_special_tokens': ['<m>', '</m>', '<h>', '</h>']}
    tokenizer.add_special_tokens(special_tokens)
    
    train_dataset = CompoundDataset(train_df, tokenizer)
    val_dataset = CompoundDataset(val_df, tokenizer)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    model = DebertaRegressor(model_name=model_name).to(device)
    model.encoder.resize_token_embeddings(len(tokenizer))
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps)
    criterion = nn.MSELoss()
    
    for epoch in range(epochs):
        train_loss = train_epoch(model, loader=train_loader, optimizer=optimizer, scheduler=scheduler, criterion=criterion)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {train_loss:.4f}")
        
    val_preds = evaluate_model(model, val_loader)
    metrics = compute_metrics(
        y_true_mod=val_df['Mod'].values,
        y_pred_mod=val_preds[:, 0],
        y_true_head=val_df['Head'].values,
        y_pred_head=val_preds[:, 1],
        compounds=val_df['Compound'].values
    )
    return model, metrics, val_preds
